# Software Engineering Foundations: Testing, Types, Logging, Errors, and Design

Maps to `design3.md` Phase 1.

This notebook is about the habits that make Python code maintainable under pressure. The aim is not to teach a framework. The aim is to teach practices that survive across codebases: testing, typing, logging, error handling, composition, and refactoring.

This notebook is deliberately standard-library-first. You can add tools like `pytest`, `mypy`, and `ruff` later, but the concepts should make sense even without external packages.


## How To Use This Notebook

Work through it in order.

1. Testing and what a good test actually proves
2. Type hints and interfaces
3. Logging and observability
4. Exceptions and resource handling
5. Design principles for maintainable Python
6. Refactoring under a safety net

For each section:
- read the explanation
- run the code
- change one example
- answer the interview-style questions before moving on


## 1. Testing: What Good Tests Actually Do

A good test does not just execute code. It proves a behavior.

Testing matters especially in data and platform work because many failures are silent:
- wrong normalization
- wrong aggregation
- dropped records
- bad edge-case handling

Good tests usually have three parts:
1. Arrange: prepare inputs
2. Act: call the code under test
3. Assert: verify the exact behavior you care about

You do not need a framework to understand this. A plain `assert` already teaches the core idea.


In [ ]:
from decimal import Decimal, InvalidOperation
from datetime import datetime, timezone


def validate_trade(raw: dict) -> dict:
    symbol = str(raw.get("symbol", "")).strip().upper()
    if not symbol:
        raise ValueError("symbol is missing or empty")

    try:
        price = Decimal(str(raw.get("price")))
    except (InvalidOperation, TypeError):
        raise ValueError("price is not a valid number")
    if price <= 0:
        raise ValueError("price must be positive")

    volume = int(raw.get("volume", 0))
    if volume <= 0:
        raise ValueError("volume must be positive")

    side = str(raw.get("side", "")).strip().lower()
    if side not in {"buy", "sell"}:
        raise ValueError("side must be 'buy' or 'sell'")

    ts = datetime.fromisoformat(str(raw.get("timestamp")))
    if ts.tzinfo is None:
        ts = ts.replace(tzinfo=timezone.utc)

    return {
        "symbol": symbol,
        "price": price,
        "volume": volume,
        "side": side,
        "timestamp": ts,
    }


valid = validate_trade(
    {
        "symbol": " eurusd ",
        "price": "1.0845",
        "volume": "1000",
        "side": "BUY",
        "timestamp": "2026-04-04T08:15:00+00:00",
    }
)

valid


In [ ]:
# Plain-assert style tests: the essence of testing

result = validate_trade(
    {
        "symbol": " eurusd ",
        "price": "1.0845",
        "volume": "1000",
        "side": "BUY",
        "timestamp": "2026-04-04T08:15:00+00:00",
    }
)
assert result["symbol"] == "EURUSD"
assert result["price"] == Decimal("1.0845")
assert result["side"] == "buy"

try:
    validate_trade({"symbol": "", "price": "1.0", "volume": "100", "side": "buy", "timestamp": "2026-01-01T00:00:00+00:00"})
except ValueError as exc:
    assert "symbol" in str(exc)
else:
    raise AssertionError("Expected ValueError for empty symbol")

"plain assert tests passed"


### Using `unittest` From the Standard Library

`unittest` is not the most concise test framework, but it is built in and teaches useful structure.

You should understand:
- test case classes
- `assertEqual`, `assertRaises`, and related assertions
- test isolation

In real codebases, many teams use `pytest` on top of the same basic testing ideas.


In [ ]:
import unittest


class ValidateTradeTests(unittest.TestCase):
    def test_valid_trade_returns_normalized_result(self):
        result = validate_trade(
            {
                "symbol": " eurusd ",
                "price": "1.0845",
                "volume": "1000",
                "side": "BUY",
                "timestamp": "2026-04-04T08:15:00+00:00",
            }
        )
        self.assertEqual(result["symbol"], "EURUSD")
        self.assertEqual(result["side"], "buy")

    def test_invalid_side_raises(self):
        with self.assertRaises(ValueError):
            validate_trade(
                {
                    "symbol": "EURUSD",
                    "price": "1.0845",
                    "volume": "1000",
                    "side": "hold",
                    "timestamp": "2026-04-04T08:15:00+00:00",
                }
            )


suite = unittest.defaultTestLoader.loadTestsFromTestCase(ValidateTradeTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
(result.testsRun, len(result.failures), len(result.errors))


### Testing Interview Questions

- What is the difference between arranging, acting, and asserting?
- What makes a test useful instead of just noisy?
- Why are silent data bugs often more dangerous than crashes in data systems?
- What is the difference between a unit test and an integration test?
- What should you test first: happy path, edge cases, or failures?


### Using `pytest` in Practice: Fixtures, Parametrize, and Mocking

`pytest` is the industry standard for Python testing. You should know three patterns cold.

**1. Fixtures** — shared setup objects, injected by name

```python
@pytest.fixture
def sample_trade():
    return Trade(symbol="EURUSD", price=Decimal("1.0845"), volume=1000)
```

Fixtures can have scope (`function`, `module`, `session`) and can yield for teardown:
```python
@pytest.fixture
def db_connection():
    conn = connect(...)
    yield conn
    conn.close()   # runs after test
```

**2. Parametrize** — run the same test with many inputs

```python
@pytest.mark.parametrize("symbol,valid", [
    ("EURUSD", True),
    ("", False),
    ("eurusd", False),
])
def test_symbol_validation(symbol, valid):
    ...
```

This avoids writing separate test functions for each case and produces clear failure messages per parameter.

**3. Mocking with `unittest.mock`** — replace real dependencies with controlled substitutes

```python
from unittest.mock import patch, MagicMock

with patch("mymodule.requests.get") as mock_get:
    mock_get.return_value.json.return_value = {"price": "1.0845"}
    result = fetch_quote("EURUSD")
```

`MagicMock` auto-creates attributes and returns `MagicMock` for any call — useful for injecting fake dependencies.

**Why pytest over unittest:**
- Less boilerplate (no class required)
- Plain `assert` with rich diff output
- Fixtures are more composable than setUp/tearDown
- Plugin ecosystem: `pytest-cov`, `pytest-asyncio`, `pytest-mock`


In [ ]:
# pytest-style tests demonstrated with the standard unittest.mock module
# (No pytest needed to understand the patterns — the logic is the same)
import unittest
from unittest.mock import patch, MagicMock, call
from decimal import Decimal
from dataclasses import dataclass
from typing import Protocol


@dataclass
class Trade:
    symbol: str
    price: Decimal
    volume: int


class PriceStore(Protocol):
    def save(self, trade: Trade) -> None: ...
    def get_latest(self, symbol: str) -> Decimal | None: ...


class TradingService:
    def __init__(self, store: PriceStore) -> None:
        self.store = store

    def process(self, symbol: str, price_str: str, volume: int) -> Trade:
        trade = Trade(
            symbol=symbol.strip().upper(),
            price=Decimal(price_str),
            volume=volume,
        )
        self.store.save(trade)
        return trade


# --- Fixture pattern (simulated with setUp) ---
class TestTradingService(unittest.TestCase):
    def setUp(self):
        # Fixture: create a fresh mock store per test
        self.mock_store = MagicMock(spec=PriceStore)
        self.service = TradingService(self.mock_store)

    def test_process_normalizes_symbol(self):
        trade = self.service.process(" eurusd ", "1.0845", 1000)
        self.assertEqual(trade.symbol, "EURUSD")

    def test_process_saves_to_store(self):
        self.service.process("EURUSD", "1.0845", 1000)
        # Assert the mock was called exactly once with correct argument
        self.mock_store.save.assert_called_once()
        saved = self.mock_store.save.call_args[0][0]
        self.assertEqual(saved.symbol, "EURUSD")

    def test_process_invalid_price_raises(self):
        with self.assertRaises(Exception):
            self.service.process("EURUSD", "not_a_number", 1000)

    # Parametrize pattern: test the same logic with multiple inputs
    def _assert_symbol_normalized(self, raw, expected):
        trade = self.service.process(raw, "1.0845", 100)
        self.assertEqual(trade.symbol, expected)

    def test_symbol_normalization_variants(self):
        for raw, expected in [
            (" eurusd ", "EURUSD"),
            ("usdjpy", "USDJPY"),
            ("  GBPUSD  ", "GBPUSD"),
        ]:
            with self.subTest(raw=raw):
                self._assert_symbol_normalized(raw, expected)

    # Mocking external dependency (simulated with patch)
    def test_with_patch(self):
        with patch.object(self.mock_store, 'get_latest', return_value=Decimal("1.0845")) as mock_get:
            price = self.mock_store.get_latest("EURUSD")
            mock_get.assert_called_once_with("EURUSD")
            self.assertEqual(price, Decimal("1.0845"))


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestTradingService)
result = unittest.TextTestRunner(verbosity=2).run(suite)
(result.testsRun, len(result.failures), len(result.errors))


## 2. Type Hints and Interfaces

Type hints are executable documentation for humans plus static analyzers.

Why they matter:
- clarify inputs and outputs
- reduce ambiguity at call sites
- help IDEs navigate code
- make refactors safer

Important idea:
- type hints do not change runtime behavior by themselves
- they are mainly for humans and tooling


In [ ]:
from dataclasses import dataclass
from typing import Protocol


@dataclass
class Trade:
    symbol: str
    price: Decimal
    volume: int


def notional_value(trade: Trade) -> Decimal:
    return trade.price * trade.volume


class TradeStore(Protocol):
    def write(self, trade: Trade) -> None:
        ...


class InMemoryTradeStore:
    def __init__(self) -> None:
        self.rows: list[Trade] = []

    def write(self, trade: Trade) -> None:
        self.rows.append(trade)


store = InMemoryTradeStore()
trade = Trade(symbol="EURUSD", price=Decimal("1.0845"), volume=1000)
store.write(trade)

{
    "notional": str(notional_value(trade)),
    "rows_written": len(store.rows),
}


### Type-Hint Interview Questions

- What problem do type hints solve in day-to-day engineering work?
- What is the difference between runtime behavior and static checking?
- What is a `Protocol`, and when is it better than depending on a concrete class?
- Why can a dataclass make code clearer than passing raw dictionaries everywhere?


## 3. Logging and Observability

`print()` is for local experimentation. `logging` is for programs you need to operate.

Good logs should answer:
- what happened
- when it happened
- how severe it was
- what business or technical context matters

You should know these log levels:
- `DEBUG`
- `INFO`
- `WARNING`
- `ERROR`
- `CRITICAL`

Design rule:
- log enough context to debug problems
- do not log secrets
- do not rely on manually scanning prints during incidents


In [ ]:
import io
import logging

log_buffer = io.StringIO()
handler = logging.StreamHandler(log_buffer)
handler.setFormatter(logging.Formatter("%(levelname)s %(name)s %(message)s"))

logger = logging.getLogger("trade_pipeline_demo")
logger.handlers.clear()
logger.setLevel(logging.INFO)
logger.addHandler(handler)
logger.propagate = False

logger.info("batch_started")
logger.warning("symbol_missing metadata_for=%s", "NZDUSD")
logger.error("write_failed sink=%s retry=%s", "warehouse", 1)

log_buffer.getvalue().splitlines()


### Logging Interview Questions

- Why is `logging` better than `print()` in production code?
- What is the difference between `INFO`, `WARNING`, and `ERROR`?
- What context would you log for a failed pipeline write?
- Why is logging too little dangerous? Why is logging too much dangerous?


## 4. Exceptions and Resource Handling

Good exception handling is about clarity and containment.

Rules worth learning early:
- raise specific exceptions when possible
- catch exceptions at the boundary where you can add useful context or recover
- do not swallow errors silently
- use `with` statements for resources that must be cleaned up

You should also understand:
- `try`
- `except`
- `else`
- `finally`


In [ ]:
class TradeValidationError(ValueError):
    pass


cleanup_events = []


def parse_price(raw: str) -> Decimal:
    try:
        value = Decimal(raw)
    except InvalidOperation as exc:
        raise TradeValidationError(f"invalid price: {raw!r}") from exc
    else:
        if value <= 0:
            raise TradeValidationError(f"price must be positive: {value}")
        return value
    finally:
        cleanup_events.append("cleanup_ran")


price_demo = {
    "good_price": str(parse_price("1.0845")),
    "finally_events": len(cleanup_events),
}

try:
    parse_price("abc")
except TradeValidationError as exc:
    price_demo["bad_price_error"] = str(exc)
    price_demo["finally_events_after_error"] = len(cleanup_events)

price_demo


In [ ]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "report.txt"
    with path.open("w", encoding="utf-8") as handle:
        handle.write("batch complete")
    saved_text = path.read_text(encoding="utf-8")

saved_text


### Exception Interview Questions

- When should you raise a custom exception instead of a built-in one?
- What is the purpose of `raise ... from exc`?
- What is the difference between `except` and `finally`?
- Why is `with open(...)` better than manual `open()` plus `close()`?


## 5. Design Principles That Actually Matter In Python

This section stays practical.

### Single Responsibility Principle
A function or class should have one reason to change.

### Composition Over Inheritance
Prefer assembling behavior from smaller objects over deep inheritance hierarchies.

### Dependency Injection
Pass collaborators in, instead of hard-coding them internally.

### Side-Effect Boundaries
Keep parsing, validation, and business logic separate from I/O where possible.

These principles matter because they make code easier to test, replace, and reason about.


In [ ]:
class PriceSource(Protocol):
    def fetch(self, symbol: str) -> Decimal:
        ...


class StaticPriceSource:
    def __init__(self, prices: dict[str, Decimal]) -> None:
        self.prices = prices

    def fetch(self, symbol: str) -> Decimal:
        return self.prices[symbol]


class SpreadCalculator:
    def __init__(self, source: PriceSource) -> None:
        self.source = source

    def spread_vs_reference(self, symbol: str, reference: Decimal) -> Decimal:
        market_price = self.source.fetch(symbol)
        return market_price - reference


source = StaticPriceSource({"EURUSD": Decimal("1.0845")})
calculator = SpreadCalculator(source)
calculator.spread_vs_reference("EURUSD", Decimal("1.0840"))


### Design Interview Questions

- What does dependency injection mean without any framework involved?
- Why is composition often simpler than inheritance in Python?
- What makes a function hard to test?
- Why is separating pure logic from I/O useful?


## 6. Refactoring Under A Safety Net

Refactoring means changing the structure of code without changing its observable behavior.

The safety net is tests.

Typical refactor moves:
- extract function
- extract class
- rename for clarity
- separate parsing from orchestration
- replace repeated logic with one helper

Bad refactoring pattern:
- changing structure and behavior at the same time with no verification


In [ ]:
raw_lines = [
    "EURUSD,1.0845,1000",
    "EURUSD,1.0847,2000",
    "USDJPY,145.12,500",
]


def parse_trade_line(line: str) -> Trade:
    symbol, price, volume = [part.strip() for part in line.split(",")]
    return Trade(symbol=symbol.upper(), price=Decimal(price), volume=int(volume))


def aggregate_notional(trades: list[Trade]) -> dict[str, Decimal]:
    totals: dict[str, Decimal] = {}
    for trade in trades:
        totals[trade.symbol] = totals.get(trade.symbol, Decimal("0")) + notional_value(trade)
    return totals


parsed_trades = [parse_trade_line(line) for line in raw_lines]
aggregate_notional(parsed_trades)


### Refactoring Interview Questions

- What is the difference between refactoring and feature work?
- Why are tests the safety net for refactoring?
- What are the signs that a function should be split apart?


## 7. Package Layout and `pyproject.toml`

A professional Python package uses `src` layout with `pyproject.toml` for reproducible builds and tooling configuration.

**Standard `src` layout:**
```
my_project/
├── pyproject.toml
├── src/
│   └── trading/
│       ├── __init__.py
│       ├── models.py
│       ├── pipeline.py
│       └── store.py
└── tests/
    ├── conftest.py       # shared fixtures
    ├── unit/
    │   └── test_pipeline.py
    └── integration/
        └── test_store.py
```

**Why `src` layout?**  
Without it, Python can import from your local directory during tests instead of the installed package — hiding install-time failures.

**`pyproject.toml` essentials:**

```toml
[build-system]
requires = ["setuptools>=70", "wheel"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "trading"
version = "0.1.0"
requires-python = ">=3.12"
dependencies = ["structlog", "pydantic>=2"]

[project.optional-dependencies]
dev = ["pytest", "pytest-cov", "mypy", "ruff"]

[tool.ruff]
line-length = 100
select = ["E", "W", "F", "I", "B"]

[tool.mypy]
strict = true

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "--cov=src --cov-report=term-missing"
```

**Key commands:**
```bash
pip install -e ".[dev]"   # editable install with dev deps
pytest tests/unit -v       # run unit tests
ruff check src/            # lint
mypy src/ --strict         # type check
```

**`conftest.py`** is where you put shared fixtures available to all tests:
```python
# tests/conftest.py
import pytest
from decimal import Decimal
from trading.models import Trade

@pytest.fixture
def sample_trade():
    return Trade(symbol="EURUSD", price=Decimal("1.0845"), volume=1000)
```

**Interview questions:**
- Why is `src` layout safer than putting your package in the root?
- What is an editable install (`pip install -e`)?
- What is `conftest.py` and why is it special in pytest?
- How would you configure code coverage thresholds in CI?


## Mini Lab

Build a tiny trade-processing module that does all of the following:
1. Parse raw rows into a typed object
2. Validate price and volume
3. Log failures with useful context
4. Store valid rows through an injected dependency
5. Cover the core behavior with tests

Interview framing:
- Where would you place the side-effect boundary?
- What would you unit-test directly?
- What would you integration-test later?


## Exit Checklist

Do not move on until you can answer these without searching:

**Testing:**
- What makes a test useful instead of just noisy?
- What is the difference between a unit test and an integration test?
- What is the Arrange-Act-Assert pattern?
- Why are silent data bugs often more dangerous than crashes?

**pytest:**
- What is a fixture in pytest and how is it different from setUp?
- What does `@pytest.mark.parametrize` do and when should you use it?
- What is `MagicMock` and what does it do when you access any attribute on it?
- How does `patch` differ from passing a mock directly?
- What is `conftest.py` and why is it special?

**Type hints:**
- What problem do type hints solve in day-to-day engineering?
- What is a `Protocol`, and when is it better than a concrete class?
- Why can a dataclass make code clearer than passing raw dictionaries?

**Logging:**
- Why is `logging` better than `print()` in production code?
- What is the difference between `INFO`, `WARNING`, and `ERROR`?

**Exceptions:**
- When should you raise a custom exception instead of a built-in one?
- What is the purpose of `raise ... from exc`?
- What is the difference between `except` and `finally`?

**Design:**
- What does dependency injection mean in plain Python?
- Why is composition often better than inheritance for simple service design?
- What is the difference between refactoring and feature work?

**Packaging:**
- Why is `src` layout safer than putting your package in the root directory?
- What is an editable install?
- What does `conftest.py` do in a pytest project?
- What goes in `[project.optional-dependencies]`?

## Official References Used To Build This Notebook

- Python Standard Library: `unittest`
  https://docs.python.org/3/library/unittest.html
- Python Standard Library: `unittest.mock`
  https://docs.python.org/3/library/unittest.mock.html
- Python Standard Library: `typing`
  https://docs.python.org/3/library/typing.html
- Python Standard Library: `dataclasses`
  https://docs.python.org/3/library/dataclasses.html
- Python Standard Library: `logging`
  https://docs.python.org/3/library/logging.html
- Python Logging HOWTO
  https://docs.python.org/3/howto/logging.html
- Python Standard Library: Built-in Exceptions
  https://docs.python.org/3/library/exceptions.html
- Python Standard Library: `contextlib`
  https://docs.python.org/3/library/contextlib.html
- PyPA: `pyproject.toml` specification
  https://packaging.python.org/en/latest/specifications/pyproject-toml/
- pytest documentation
  https://docs.pytest.org/en/stable/